# C6 — Standard split vs. mockup-free generic split

Loads every architecture that `020_bis_training_generic.ipynb` actually trained
(any `models/deterministic_generic/<arch>/best_model.keras` that exists) and
scores it side by side against the same architecture's standard-split checkpoint
in `models/deterministic/<arch>/`, on the **same `data/test/` images** — the
held-out GT-annotated paintings that neither split ever touches.

| | `standard` | `generic` |
|---|---|---|
| checkpoints | `models/deterministic/<arch>/` | `models/deterministic_generic/<arch>/` |
| trained by | `020_training.ipynb` | `020_bis_training_generic.ipynb` |
| mockups in training data | yes (pair-level) | **no** |
| artwork grouping | yes | **no** |

Same three axes as the C0–C5 / 04x–05x series: **reconstruction fidelity**
(`mae`/`ssim`/`psnr` of the prediction against the real IR), **detection AUROC**
of the delta signals against the hand-drawn masks (`GT01`/`GT02`/`GT03` only),
and **stroke coherence** (reference-free, all images). Deterministic
architectures only — the NLL head is out of scope here.

`gain` is always oriented so **positive means the generic split improved** over
standard.

Make the project root importable so `scripts.*` resolves regardless of the
notebook's working directory.

In [1]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports — the same building blocks as the 04x/05x series: `scripts.delta_analysis`
for the delta/structural maps, `scripts.detection` for AUROC against the masks,
`scripts.stroke_stats` for the reference-free corroboration.

In [2]:
import gc

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from scripts.config import settings
from scripts.delta_analysis import analyze_delta
from scripts.detection import evaluate_detection
from scripts.stroke_stats import stroke_coherence
from scripts.trainer import load_model

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. The `data/test/` images

Every `(rgb, ir)` pair under `data/test/`, discovered generically. The three
with a hand-drawn mask in `data/test/annotations/*_Map.png` (`GT01`, `GT02`,
`GT03`) additionally carry a detection ground truth; §4's AUROC is restricted to
those, everything else runs on all of them.

In [3]:
TEST_RGB_DIR = project_root / "data" / "test" / "rgb"
TEST_IR_DIR = project_root / "data" / "test" / "ir"
ANNOTATIONS_DIR = project_root / "data" / "test" / "annotations"

rgb_paths = sorted(TEST_RGB_DIR.glob("*.jpg")) + sorted(TEST_RGB_DIR.glob("*.png"))
image_pairs = [
    (p, TEST_IR_DIR / p.name)
    for p in sorted(rgb_paths)
    if (TEST_IR_DIR / p.name).exists()
]
gt_stems = {
    p.name.removesuffix("_Map.png") for p in sorted(ANNOTATIONS_DIR.glob("*_Map.png"))
}
gt_stems &= {p.stem for p, _ in image_pairs}

print(f"data/test/ images: {len(image_pairs)} | {[p.stem for p, _ in image_pairs]}")
print(f"with a ground-truth mask: {sorted(gt_stems)}")

if not image_pairs:
    raise RuntimeError("No (rgb, ir) pairs found under data/test/.")

data/test/ images: 10 | ['GT01', 'GT02', 'GT03', 'bridge', 'case', 'face', 'green', 'modern', 'modern2', 'total']
with a ground-truth mask: ['GT01', 'GT02', 'GT03']


## 2. Which architectures were trained on the generic split

`VERSIONS` says where each split's checkpoints live. The compared set is
**auto-discovered**: every architecture with a `best_model.keras` under
`models/deterministic_generic/` that *also* has one under `models/deterministic/`.
Comment architectures in/out of `020_bis_training_generic.ipynb` and this list
follows — no edit here needed.

In [4]:
VERSIONS = {
    "standard": settings.MODELS_DIR / "deterministic",
    "generic": settings.MODELS_DIR / "deterministic_generic",
}

# Any deterministic architecture 020 / 020_bis can train.
CANDIDATE_ARCHS = [
    "unet",
    "resunet",
    "attention_unet",
    "unet_residual",
    "unet_v2",
    "unet_restormer",
]


def has_ckpt(root: Path, arch: str) -> bool:
    return (root / arch / "best_model.keras").exists()


ARCHS = [
    arch
    for arch in CANDIDATE_ARCHS
    if has_ckpt(VERSIONS["generic"], arch) and has_ckpt(VERSIONS["standard"], arch)
]

only_generic = [a for a in CANDIDATE_ARCHS
                if has_ckpt(VERSIONS["generic"], a) and not has_ckpt(VERSIONS["standard"], a)]

print(f"comparing: {ARCHS}")
if only_generic:
    print(f"[skip] trained on generic split but no standard-split pair: {only_generic}")
if not ARCHS:
    raise RuntimeError(
        "No architecture has a checkpoint in both models/deterministic_generic/ "
        "and models/deterministic/. Run 020_bis_training_generic.ipynb first."
    )

comparing: ['unet', 'resunet', 'attention_unet']


## 3. Signals and the per-image sweep

Identical construction to `040`/`052`: for each model, predict the full IR image,
then `analyze_delta` gives the **raw delta** `|real_IR − pred|` and the
**structural delta** `1 − local SSIM structure`. Fidelity is `mae`/`ssim`/`psnr`
of the prediction against the real IR. Detection AUROC scores each delta against
the mask (GT images only); stroke coherence runs on every image.

One architecture at a time, both split-versions held together so their signals
come from the exact same image arrays, then both released before the next.

In [5]:
MASK_THRESHOLD = 127  # midpoint threshold for the mask's anti-aliased edges
SIGNAL_KINDS = ("raw delta", "structural delta")


def load_pair(rgb_path: Path, ir_path: Path) -> tuple[np.ndarray, np.ndarray]:
    rgb = np.array(Image.open(rgb_path).convert("RGB")).astype(np.float32) / 255.0
    ir = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    return rgb, ir


def load_mask(stem: str) -> np.ndarray:
    mask_path = ANNOTATIONS_DIR / f"{stem}_Map.png"
    return np.array(Image.open(mask_path).convert("L")) > MASK_THRESHOLD


def pad_to_multiple(arr: np.ndarray, multiple: int = 16) -> tuple[np.ndarray, tuple[int, int]]:
    h, w = arr.shape[:2]
    ph, pw = (-h) % multiple, (-w) % multiple
    padded = np.pad(arr, ((0, ph), (0, pw), (0, 0)))
    return padded, (h, w)


def predict_ir(model: tf.keras.Model, rgb: np.ndarray) -> np.ndarray:
    """Full-image IR prediction, cropped back to the original size."""
    padded, (h, w) = pad_to_multiple(rgb)
    pred = model.predict(padded[np.newaxis], verbose=0)[0]
    return np.clip(pred[:h, :w, 0], 0.0, 1.0)


def signals_for(real_ir: np.ndarray, pred_ir: np.ndarray) -> dict[str, np.ndarray]:
    res = analyze_delta(real_ir, pred_ir)
    return {
        "raw delta": res.raw_delta,
        "structural delta": res.structural_delta,
    }


def fidelity(real_ir: np.ndarray, pred_ir: np.ndarray) -> dict[str, float]:
    real = tf.constant(real_ir[..., np.newaxis])
    pred = tf.constant(pred_ir[..., np.newaxis])
    return {
        "mae": float(np.mean(np.abs(real_ir - pred_ir))),
        "ssim": float(tf.image.ssim(real, pred, max_val=1.0)),
        "psnr": float(tf.image.psnr(real, pred, max_val=1.0)),
    }

In [6]:
fidelity_rows: dict[tuple[str, str], dict[str, list[float]]] = {}
auroc_rows: dict[tuple[str, str, str], dict[str, float]] = {}
coherence_rows: dict[tuple[str, str, str], list[float]] = {}

for arch in ARCHS:
    loaded: dict[str, tf.keras.Model] = {}
    for version, root in VERSIONS.items():
        loaded[version] = load_model(arch, model_dir=root)

    print(f"\n=== {arch} — {' vs. '.join(loaded)} ===")

    for rgb_path, ir_path in image_pairs:
        stem = rgb_path.stem
        rgb, real_ir = load_pair(rgb_path, ir_path)
        mask = load_mask(stem) if stem in gt_stems else None

        for version, model in loaded.items():
            pred_ir = predict_ir(model, rgb)
            sigs = signals_for(real_ir, pred_ir)

            fid = fidelity(real_ir, pred_ir)
            row = fidelity_rows.setdefault((arch, version), {k: [] for k in fid})
            for k, v in fid.items():
                row[k].append(v)

            for kind, sig in sigs.items():
                coherence_rows.setdefault((arch, version, kind), []).append(
                    stroke_coherence(sig).coherence
                )
                if mask is not None:
                    det = evaluate_detection(sig, mask)
                    auroc_rows.setdefault((arch, version, kind), {})[stem] = det.auroc

    del loaded
    gc.collect()
    tf.keras.backend.clear_session()

2026-08-29 16:15:01.250936: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Max
2026-08-29 16:15:01.250968: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 36.00 GB
2026-08-29 16:15:01.250979: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 14.04 GB
2026-08-29 16:15:01.251003: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-29 16:15:01.251021: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



=== unet — standard vs. generic ===


2026-08-29 16:15:02.320881: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.



=== resunet — standard vs. generic ===

=== attention_unet — standard vs. generic ===


## 4. Reconstruction fidelity

`mae`/`ssim`/`psnr` of the prediction against the real IR, averaged over all
`data/test/` images. `gain > 0` means the generic split predicts the IR better.

In [7]:
LOWER_IS_BETTER = {"mae": True, "ssim": False, "psnr": False}


def gain(metric: str, std: float, gen: float) -> float:
    return std - gen if LOWER_IS_BETTER[metric] else gen - std


col_w = 22
header = "architecture".ljust(col_w)
for metric in LOWER_IS_BETTER:
    header += f"{metric} std".rjust(13) + f"{metric} gen".rjust(13) + f"{metric} gain".rjust(13)
print(header)
print("-" * len(header))

for arch in ARCHS:
    row = arch.ljust(col_w)
    for metric in LOWER_IS_BETTER:
        std = float(np.mean(fidelity_rows[(arch, "standard")][metric]))
        gen = float(np.mean(fidelity_rows[(arch, "generic")][metric]))
        row += f"{std:.4f}".rjust(13) + f"{gen:.4f}".rjust(13)
        row += f"{gain(metric, std, gen):+.4f}".rjust(13)
    print(row)

print(f"\n(mean over {len(image_pairs)} data/test/ images; gain > 0 means generic better)")

architecture                mae std      mae gen     mae gain     ssim std     ssim gen    ssim gain     psnr std     psnr gen    psnr gain
-------------------------------------------------------------------------------------------------------------------------------------------
unet                         0.1265       0.1160      +0.0106       0.6071       0.6165      +0.0094      16.6486      17.1467      +0.4981
resunet                      0.1084       0.1057      +0.0027       0.6121       0.6218      +0.0097      17.6901      17.7268      +0.0367
attention_unet               0.1244       0.1960      -0.0716       0.6210       0.5323      -0.0886      16.7330      13.2280      -3.5049

(mean over 10 data/test/ images; gain > 0 means generic better)


## 5. Detection AUROC — the signal that matters

AUROC of each delta signal against the hand-drawn masks, per GT image and
averaged. `0.5` is chance. With `n = 3` the mean hides a lot, so the per-image
breakdown follows.

In [8]:
if not auroc_rows:
    print("No ground-truth mask found under data/test/annotations/ — section skipped.")
else:
    stems = sorted(gt_stems)
    name_w = 40
    header = "signal".ljust(name_w) + "std".rjust(10) + "gen".rjust(10) + "gain".rjust(10)
    print(header)
    print("-" * len(header))
    for arch in ARCHS:
        for kind in SIGNAL_KINDS:
            key_s, key_g = (arch, "standard", kind), (arch, "generic", kind)
            if key_s not in auroc_rows or key_g not in auroc_rows:
                continue
            std = float(np.mean(list(auroc_rows[key_s].values())))
            gen = float(np.mean(list(auroc_rows[key_g].values())))
            print(f"{arch} [{kind}]".ljust(name_w)
                  + f"{std:.4f}".rjust(10) + f"{gen:.4f}".rjust(10) + f"{gen - std:+.4f}".rjust(10))

    print("\nper-image AUROC breakdown\n")
    header = "signal".ljust(name_w) + "".join(
        f"{s} std".rjust(12) + f"{s} gen".rjust(12) for s in stems
    )
    print(header)
    print("-" * len(header))
    for arch in ARCHS:
        for kind in SIGNAL_KINDS:
            if (arch, "standard", kind) not in auroc_rows:
                continue
            row = f"{arch} [{kind}]".ljust(name_w)
            for s in stems:
                for version in ("standard", "generic"):
                    row += f"{auroc_rows[(arch, version, kind)][s]:.3f}".rjust(12)
            print(row)

signal                                         std       gen      gain
----------------------------------------------------------------------
unet [raw delta]                            0.4726    0.5637   +0.0911
unet [structural delta]                     0.7073    0.7044   -0.0029
resunet [raw delta]                         0.5520    0.5357   -0.0163
resunet [structural delta]                  0.6810    0.6906   +0.0096
attention_unet [raw delta]                  0.4993    0.3996   -0.0997
attention_unet [structural delta]           0.7007    0.5548   -0.1459

per-image AUROC breakdown

signal                                      GT01 std    GT01 gen    GT02 std    GT02 gen    GT03 std    GT03 gen
----------------------------------------------------------------------------------------------------------------
unet [raw delta]                               0.442       0.544       0.461       0.565       0.515       0.582
unet [structural delta]                        0.815       0.804 

## 6. Stroke coherence — reference-free corroboration

How much of each signal is oriented, line-like structure rather than isotropic
noise (`scripts.stroke_stats`). No mask needed, so this runs on **all**
`data/test/` images and is an independent check on the AUROC verdict.

In [9]:
name_w = 40
print("signal".ljust(name_w) + "std".rjust(16) + "gen".rjust(16) + "gain".rjust(12))
print("-" * (name_w + 44))
for arch in ARCHS:
    for kind in SIGNAL_KINDS:
        std = np.array(coherence_rows[(arch, "standard", kind)])
        gen = np.array(coherence_rows[(arch, "generic", kind)])
        print(f"{arch} [{kind}]".ljust(name_w)
              + f"{std.mean():.3f}±{std.std():.3f}".rjust(16)
              + f"{gen.mean():.3f}±{gen.std():.3f}".rjust(16)
              + f"{gen.mean() - std.mean():+.4f}".rjust(12))
print(f"\n(mean ± std over {len(image_pairs)} data/test/ images)")

signal                                               std             gen        gain
------------------------------------------------------------------------------------
unet [raw delta]                             0.244±0.069     0.256±0.082     +0.0125
unet [structural delta]                      0.357±0.065     0.368±0.060     +0.0109
resunet [raw delta]                          0.244±0.068     0.252±0.075     +0.0085
resunet [structural delta]                   0.351±0.062     0.362±0.060     +0.0105
attention_unet [raw delta]                   0.253±0.079     0.288±0.064     +0.0359
attention_unet [structural delta]            0.373±0.071     0.428±0.055     +0.0556

(mean ± std over 10 data/test/ images)


## 7. Verdict

A coarse per-architecture tally of how many measured quantities the generic
split improved — fidelity (`mae`/`ssim`/`psnr`), detection AUROC (both deltas,
mean over GT images), coherence (both deltas). It collapses axes that need not
agree; read §4–§6 for what actually moved.

In [10]:
def tally(gains: list[float]) -> str:
    if not gains:
        return "-"
    return f"{sum(g > 0 for g in gains)}/{len(gains)}"

name_w = 22
print("architecture".ljust(name_w) + "fidelity".rjust(12) + "detection".rjust(12) + "coherence".rjust(12))
print("-" * (name_w + 36))
for arch in ARCHS:
    fid_gains = [
        gain(m, float(np.mean(fidelity_rows[(arch, "standard")][m])),
             float(np.mean(fidelity_rows[(arch, "generic")][m])))
        for m in LOWER_IS_BETTER
    ]
    det_gains = [
        float(np.mean(list(auroc_rows[(arch, "generic", k)].values())))
        - float(np.mean(list(auroc_rows[(arch, "standard", k)].values())))
        for k in SIGNAL_KINDS
        if (arch, "standard", k) in auroc_rows
    ]
    coh_gains = [
        float(np.mean(coherence_rows[(arch, "generic", k)]))
        - float(np.mean(coherence_rows[(arch, "standard", k)]))
        for k in SIGNAL_KINDS
    ]
    print(arch.ljust(name_w) + tally(fid_gains).rjust(12)
          + tally(det_gains).rjust(12) + tally(coh_gains).rjust(12))

print("\n(generic better / total measured)")

architecture              fidelity   detection   coherence
----------------------------------------------------------
unet                           3/3         1/2         2/2
resunet                        3/3         1/2         2/2
attention_unet                 0/3         0/2         2/2

(generic better / total measured)
